In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

from src.problems import (
    MMF1, MMF4, MMF11_L, MMF16_L3, MMF16_20,
    ZDT1, ZDT3, ZDT4, ZDT6,
    DTLZ1, DTLZ2, DTLZ3, DTLZ4, DTLZ7,
    WFG1, WFG2, WFG4, WFG5, WFG9,
    BBOB_F1_Sphere_Sphere, 
    BBOB_F5_Sphere_SharpRidge,
    BBOB_F17_EllipsoidSeparable_SchafferF7,
    BBOB_F22_AttractiveSector_SharpRidge,
    BBOB_F37_SharpRidge_Rastrigin, 
    BBOB_F49_Rastrigin_Gallagher101,
    BBOB_F55_Gallagher101_Gallagher101,
    evaluate_problem,
)
from src.visualization import display_pareto_fronts_catalog, display_decision_space

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

# Efficient Non-dominated Sort (ENS, Roy et al. 2016): O(N * log^(M-1) N) e
# sem matriz NxN - necessario para landscapes grandes (N >> 1e5).
_NDS = NonDominatedSorting(method="efficient_non_dominated_sort")

In [ ]:
# Global settings
carrega_resultados = True # se true tenta carregar os parquets da landscape e do pareto empirico - se nao acha, calcula (avaliado individualmente)
N_SAMPLES = 2_000_000

DATA_DIR = 'data/dataframes'
VALID_METHODS = ['sobol', 'latin_hypercube', 'random'] # o primeiro da lista é mostrado nos gráficos de fitness landscape
METHOD_COLORS = {'sobol': 'red', 'latin_hypercube': 'dodgerblue', 'random': 'orange'}
PLOT_3D_PCA = ['MMF16_20','DTLZ1','DTLZ2','DTLZ3','DTLZ4','DTLZ7']
OUT_2D_PCA = ['MMF16_20']

# Fitness Landscape – Catálogo Completo de Problemas

Cada classe de problema possui o método `true_pareto_front(n)` que retorna `(X, F)`.

A função `display_pareto_fronts_catalog(problem, F_landscape, ...)` mostra:
- **Cinza**: amostra do espaço de objetivos
- **Vermelho**: Pareto front teórico verdadeiro (do método da classe)
- **Azul**: Pareto front empírico (non-dominated sorting sobre os dados amostrados)

In [ ]:
################################################################## 
#################  Amostragem Fitness Landscape  #################

def generate_samples(problem, n_samples=N_SAMPLES, method='sobol'):
    """Sample the decision space and evaluate objectives.

    Uses a full grid for n_var <= 3 (regardless of method).
    For n_var > 3:
      'sobol'           - scrambled Sobol sequence; generates exactly 2^ceil(log2(n_samples)) points.
      'latin_hypercube' - Latin Hypercube Sampling (original behaviour).
      'random'          - uniform random sampling.
    """
    if method not in VALID_METHODS:
        raise ValueError(f"method must be one of {VALID_METHODS}, got '{method}'")
    n = problem.n_var
    if n <= 3:
        n_pts = max(2, int(round(n_samples ** (1.0 / n))))
        grids = [np.linspace(problem.xl[i], problem.xu[i], n_pts) for i in range(n)]
        mesh = np.meshgrid(*grids, indexing='ij')
        X = np.column_stack([m.ravel() for m in mesh])
    else:
        if method == 'sobol':
            m = int(np.ceil(np.log2(n_samples)))
            sampler = qmc.Sobol(d=n, scramble=True, seed=42)
            X = qmc.scale(sampler.random_base2(m), problem.xl, problem.xu)
        elif method == 'latin_hypercube':
            sampler = qmc.LatinHypercube(d=n, seed=42)
            X = qmc.scale(sampler.random(n_samples), problem.xl, problem.xu)
        elif method == 'random':
            rng = np.random.default_rng(42)
            X = rng.uniform(problem.xl, problem.xu, size=(n_samples, n))
    return X, evaluate_problem(problem, X)


def _landscape_path(out_dir, name, method):
    return os.path.join(out_dir, f'df_{name}_landscape_{method}.parquet')


def _load_or_generate_landscape(problem, name, method, n_samples, out_dir,
                                x_cols, f_cols):
    """Try to load the landscape parquet; otherwise generate and save."""
    path = _landscape_path(out_dir, name, method)
    if carrega_resultados and os.path.exists(path):
        df = pd.read_parquet(path)
        X = df[x_cols].to_numpy()
        F = df[f_cols].to_numpy()
        print(f'  Landscape carregado de {path}: {len(X):,} pontos')
    else:
        X, F = generate_samples(problem, n_samples, method=method)
        df = pd.DataFrame(np.column_stack([X, F]), columns=x_cols + f_cols)
        df.to_parquet(path)
        print(f'  Landscape gerado e salvo em {path}: {len(X):,} pontos')
    return X, F



################################################################## 
#################  Cálculo Pareto Front Empírico  ################

def _pf_empirico_path(out_dir, name, method):
    return os.path.join(out_dir, f'df_{name}_pf_empirico_{method}.parquet')


def _load_or_compute_pf(X, F, name, method, out_dir, x_cols, f_cols):
    """Try to load the empirical PF parquet; otherwise compute via NDS and save."""
    path = _pf_empirico_path(out_dir, name, method)
    if carrega_resultados and os.path.exists(path):
        df_pf = pd.read_parquet(path)
        pf_X = df_pf[x_cols].to_numpy()
        pf_F = df_pf[f_cols].to_numpy()
        print(f'  PF empirico carregado de {path}: {len(pf_F)} pontos')
    else:
        pf_idx = _NDS.do(F, only_non_dominated_front=True)
        pf_X = X[pf_idx]
        pf_F = F[pf_idx]
        df_pf = pd.DataFrame(np.column_stack([pf_X, pf_F]), columns=x_cols + f_cols)
        df_pf.to_parquet(path)
        print(f'  PF empirico calculado e salvo em {path}: {len(pf_F)} pontos')
    return pf_X, pf_F




################################################################## 
######################     Fluxo Completo    #####################

def analyze_problem(problem, name, n_samples=100_000,
                    methods=None,
                    pair_vars=None,
                    true_color='white',
                    elev_offset=-15, azim_offset=225, roll_offset=0,
                    **plot_kwargs):
    """
    Pipeline completo de analise para um problema do catalogo.

    Para cada metodo de amostragem em `methods`:
        1. Amostra do espaco de decisao:
           - se carrega_resultados=True e o parquet existir, le de
             data/dataframes/{name}/df_{name}_landscape_{method}.parquet
           - caso contrario, gera (grid para n_var<=3, metodo escolhido
             caso contrario) e salva.
        2. Pareto front empirico:
           - se carrega_resultados=True e o parquet existir, le de
             data/dataframes/{name}/df_{name}_pf_empirico_{method}.parquet
           - caso contrario, calcula via non-dominated sorting (ENS) a
             partir da amostra e salva.
    Apos todos os metodos:
        3. PF teorico via problem.true_pareto_front()
        4. display_pareto_fronts_catalog com um PF empirico por metodo
        5. display_decision_space com os dados do primeiro metodo

    Parameters
    ----------
    problem : Problem
        Instancia de problema do catalogo (deve ter true_pareto_front()).
    name : str
        Nome do problema (usado como titulo e nome da pasta de saida).
    n_samples : int, default 100_000
        Numero de amostras a gerar por metodo. Para Sobol, o numero real
        e 2^ceil(log2(n_samples)).
    methods : list of str or None, default None
        Metodos de amostragem a utilizar. Valores validos: 'sobol',
        'latin_hypercube', 'random'. None = todos os tres.
    pair_vars : list of int or None, default None
        Variaveis de decisao (1-indexed) a incluir no grid pairwise.
        Ex: [1, 2, 20] -> grid 3x3 com x1, x2, x3.
        None = grid completo (todas as variaveis).
    true_color : str, default 'white'
        Cor dos pontos/linha do Pareto front/set teorico.
        'white' = preenchimento branco com contorno preto.
    elev_offset : float, default -15
        Offset de elevacao (eixo X) para graficos 3D.
        Elevacao final = 25 + elev_offset.
    azim_offset : float, default 225
        Offset de azimute (eixo Z) para graficos 3D.
        Azimute final = 45 + azim_offset.
    roll_offset : float, default 0
        Offset de roll (eixo Y) para graficos 3D.
    **plot_kwargs
        Parametros adicionais repassados a display_pareto_fronts_catalog
        (ex: xlim, ylim, zlim para limites manuais dos eixos).
    """
    ########################## Cálculos ##########################

    # Inicializa parametros
    if methods is None:
        methods = list(VALID_METHODS)

    x_cols = [f'x_{i+1}' for i in range(problem.n_var)]
    f_cols = [f'f{i+1}'  for i in range(problem.n_obj)]
    results = []
    X_first, F_first, pf_X_first = None, None, None
    
    # Cria pasta de outpus se não existir
    out_dir = os.path.join(DATA_DIR, name)
    os.makedirs(out_dir, exist_ok=True)


    # Amostra landscape e calcula pareto empírico para cada método de amostragem
    for method in methods:
        print(f'\n[{method}] Processando {name} '
              f'(n_var={problem.n_var}, n_obj={problem.n_obj})...')

        # Amostra landscape
        X, F = _load_or_generate_landscape(
            problem, name, method, n_samples, out_dir, x_cols, f_cols)
        
        # Calcula pareto empírico
        pf_emp_X, pf_emp_F = _load_or_compute_pf(
            X, F, name, method, out_dir, x_cols, f_cols)

        # o front do primeiro método da lista é mostrado no gráfico de landscape
        if X_first is None:
            X_first, F_first, pf_X_first = X, F, pf_emp_X

        results.append({'method': method, 'pf_F': pf_emp_F, 'pf_X': pf_emp_X})

    # Obtém pareto real
    X_true, F_true = problem.true_pareto_front()
    print(f'\n  PF teorico: {len(F_true)} pontos')


    ########################## Plots ##########################

    # Configura rotação default de gráficos 3D
    rot = dict(elev_offset=elev_offset, azim_offset=azim_offset, roll_offset=roll_offset)

    # Print front pareto
    print('\n  -- Espaco de Objetivos --')
    display_pareto_fronts_catalog(
        problem, F_first,
        [r['pf_F'] for r in results],
        front_names=[f"PF {r['method']}" for r in results],
        front_colors=[METHOD_COLORS[r['method']] for r in results],
        sample_size=100_000,
        title=name,
        true_color=true_color,
        **rot, **plot_kwargs,
    )

    # Print fitness landscape
    print('  -- Espaco de Decisoes --')
    first_color = METHOD_COLORS[results[0]['method']]
    display_decision_space(problem, X_first, F_first, X_true, pf_X_first, title=name,
                           pair_vars=pair_vars,
                           true_color=true_color, 
                           emp_color=first_color,
                           show_2d_pca=False if name in OUT_2D_PCA else True,
                           show_3d_pca=True if name in PLOT_3D_PCA else False,
                           **rot)


---
## MMF – Multimodal Multi-objective Functions (CEC 2020)

### MMF1
Convex PF, 2 global PSs espelhados em x₁ = 2.  
PF: f₂ = 1 − √f₁

In [ ]:
analyze_problem(MMF1(), 'MMF1')

### MMF4
Concave PF, 4 global PSs (niching simétrico).  
PF: f₂ = 1 − f₁²

In [ ]:
analyze_problem(MMF4(), 'MMF4')

### MMF11_L
1 global PS + 1 local PS.  
PF: f₂ = g*/f₁ (hipérbole)

In [ ]:
analyze_problem(MMF11_L(), 'MMF11_L')

### MMF16_L3
3-objetivo, 3 variáveis, 2 global PS + 2 local PS.  
PF: esfera de raio (1 + g*)

In [ ]:
analyze_problem(MMF16_L3(), 'MMF16_L3', 
                pair_vars=None, #Variáveis de decisão (1-indexed) a incluir no grid pairwise. Ex: [1, 2, 20] → grid 3x3 com x1, x2, x3. None = grid completo (todas as variáveis).
                true_color='red', #Cor dos pontos/linha do Pareto front/set teórico. 'white' = preenchimento branco com contorno preto.
                emp_color='white', #Cor dos pontos do Pareto front/set empírico.
                elev_offset=-27, #Offset de elevação (eixo X) para gráficos 3D. Elevação final = 25 + elev_offset.
                azim_offset=225, #Offset de azimute (eixo Z) para gráficos 3D. Azimute final = 45 + azim_offset.
                roll_offset=0) #Offset de roll (eixo Y) para gráficos 3D.
                #Parâmetros adicionais repassados a display_pareto_fronts_catalog (ex: xlim, ylim, zlim para limites manuais dos eixos).

### MMF16_20
3-objetivo, 20 variáveis (só x₂₀ participa de g).  
PF: esfera de raio (1 + g*)

In [ ]:
#analyze_problem(MMF16_20(), 'MMF16_20', pair_vars=[1, 2, 20])

---
## ZDT – Zitzler, Deb, Thiele (2000)

### ZDT1
Convex PF, 30 variáveis.  
PF: f₂ = 1 − √f₁

In [ ]:
analyze_problem(ZDT1(), 'ZDT1', 
                pair_vars=[1, 2], 
                true_color='white', 
                emp_color='red',
                elev_offset=-27, 
                azim_offset=225, 
                roll_offset=0)

### ZDT3
Disconnected PF (5 segmentos), 30 variáveis.  
PF: f₂ = 1 − √f₁ − f₁·sin(10πf₁)

In [ ]:
analyze_problem(ZDT3(), 'ZDT3', pair_vars=[1, 2])

### ZDT4
Multimodal g (muitos fronts locais), 10 variáveis.  
PF: f₂ = 1 − √f₁

In [ ]:
analyze_problem(ZDT4(), 'ZDT4', 
                #ylim=(0, 5), 
                pair_vars=[1, 2])

### ZDT6
Nonconvex, non-uniform density, 10 variáveis.  
PF: f₂ = 1 − f₁²

In [ ]:
analyze_problem(ZDT6(), 'ZDT6', pair_vars=[1, 2])

---
## DTLZ – Deb, Thiele, Laumanns, Zitzler (2001)

### DTLZ1
Linear PF, highly multimodal g, 7 variáveis.  
PF: f₁ + f₂ + f₃ = 0.5

In [ ]:
analyze_problem(DTLZ1(), 'DTLZ1',  # para visualizar o true pareto com qualidade, limitar tudo em (0,2)
                xlim=(0, 100),  # (0, 2),
                ylim=(0, 100),  # (0, 2),
                zlim=(0, 100),  # (0, 2),
                pair_vars=[1, 2, 3])

### DTLZ2
Spherical PF, 12 variáveis.  
PF: f₁² + f₂² + f₃² = 1

In [ ]:
analyze_problem(DTLZ2(), 'DTLZ2', 
                xlim=(0, 2),
                ylim=(0, 2),
                zlim=(0, 2),
                pair_vars=[1, 2, 3])

### DTLZ3
Spherical PF + DTLZ1’s multimodal g, 12 variáveis.  
PF: f₁² + f₂² + f₃² = 1

In [ ]:
analyze_problem(DTLZ3(), 'DTLZ3', # para visualizar o true pareto com qualidade, limitar tudo em (0,2)
                xlim=(0, 1000), # (0, 2),
                ylim=(0, 1000), # (0, 2),
                zlim=(0, 1000), #  (0, 2),
                pair_vars=[1, 2, 3])

### DTLZ4
Biased density on spherical PF (α = 100), 12 variáveis.  
PF: f₁² + f₂² + f₃² = 1

In [ ]:
analyze_problem(DTLZ4(), 'DTLZ4', 
                xlim=(0, 2),
                ylim=(0, 2),
                zlim=(0, 2),
                pair_vars=[1, 2, 3])

### DTLZ7
Disconnected PF (4 patches), 22 variáveis.  
PF: f₃ = 6 − f₁(1+sin(3πf₁)) − f₂(1+sin(3πf₂))

In [ ]:
analyze_problem(DTLZ7(), 'DTLZ7', 
                zlim=(0, 20),
                pair_vars=[1, 2, 3])

---
## WFG – Walking Fish Group (Huband et al. 2006)

### WFG1
Separable, unimodal, biased. PF: convex + mixed.

In [ ]:
analyze_problem(WFG1(), 'WFG1', pair_vars=[1, 2, 3])

### WFG2
Non-separable distance. PF: convex + disconnected.

In [ ]:
analyze_problem(WFG2(), 'WFG2', pair_vars=[1, 2, 3])

### WFG4
Multimodal (s_multi on every param). PF: concave.

In [ ]:
analyze_problem(WFG4(), 'WFG4', pair_vars=[1, 2, 3])

### WFG5
Deceptive (s_decept on every param). PF: concave.

In [ ]:
analyze_problem(WFG5(), 'WFG5', pair_vars=[1, 2, 3])

### WFG9
Parameter-dependent bias, deceptive + multimodal, nonseparable. PF: concave.

In [ ]:
analyze_problem(WFG9(), 'WFG9', pair_vars=[1, 2, 3])

---
## BBOB bi-objetivo (bbob-biobj, Tušar et al. 2016)

7 funções da suíte oficial **bbob-biobj**, cada uma combinando duas funções single-objective bbob (Hansen et al. 2009). Objetivos crus (sem normalização), `f_opt ≡ 0`, instâncias distintas por objetivo (Kα=2K+1, Kβ=2K+2), domínio `[-5, 5]^10`.

PF de referência: **F1 Sphere/Sphere é analítica** (segmento entre os dois ótimos, front convexo, HV normalizado fechado = 0.8333). As outras 6 não têm forma fechada — `true_pareto_front()` roda NSGA-II acumulado (orçamento padrão pop=200, n_gen=300, 5 seeds) e cacheia em `data/bbob_pf_cache/`. **A 1ª execução de cada uma é lenta**; execuções seguintes leem o cache.

### F1 – Sphere / Sphere
bbob f1 / f1. PF **analítica**: segmento entre os dois ótimos esféricos, front convexo.

In [ ]:
analyze_problem(BBOB_F1_Sphere_Sphere(), 'BBOB1',
                xlim = (0,210),
                ylim = (0,210),
                pair_vars=[1, 2, 3, 4],
                true_color='white',
                emp_color='red',
                elev_offset=-27,
                azim_offset=225,
                roll_offset=0)

### F5 – Sphere / Sharp ridge
bbob f1 / f13. PF empírica (NSGA-II).

In [ ]:
analyze_problem(BBOB_F5_Sphere_SharpRidge(), 'BBOB5',
                xlim = (0,220),
                ylim = (0,3200),
                pair_vars=[1, 2, 3, 4],
                true_color='white',
                emp_color='red',
                elev_offset=-27,
                azim_offset=225,
                roll_offset=0)

### F17 – Ellipsoid separable / Schaffer F7 cond 10
bbob f2 / f17. PF empírica (NSGA-II).

In [ ]:
analyze_problem(BBOB_F17_EllipsoidSeparable_SchafferF7(), 'BBOB17',
                xlim = (0,5_000_000),
                ylim = (0,200),
                pair_vars=[1, 2, 3, 4],
                true_color='white',
                emp_color='red',
                elev_offset=-27,
                azim_offset=225,
                roll_offset=0)

### F22 – Attractive sector / Sharp ridge
bbob f6 / f13. PF empírica (NSGA-II).

In [ ]:
analyze_problem(BBOB_F22_AttractiveSector_SharpRidge(), 'BBOB22',
                xlim = (0,500_000),
                ylim = (0,2000),
                pair_vars=[1, 2, 3, 4],
                true_color='white',
                emp_color='red',
                elev_offset=-27,
                azim_offset=225,
                roll_offset=0)

### F37 – Sharp ridge / Rastrigin
bbob f13 / f15. PF empírica (NSGA-II).

In [ ]:
analyze_problem(BBOB_F37_SharpRidge_Rastrigin(), 'BBOB37',
                xlim = (0,3500),
                ylim = (0,7000),
                pair_vars=[1, 2, 4, 5],
                true_color='white',
                emp_color='red',
                elev_offset=-27,
                azim_offset=225,
                roll_offset=0)

### F49 – Rastrigin / Gallagher 101 peaks
bbob f15 / f21. PF empírica (NSGA-II).

In [ ]:
analyze_problem(BBOB_F49_Rastrigin_Gallagher101(), 'BBOB49',
                xlim = (0,5000),
                pair_vars=[1, 3, 7, 9],
                true_color='white',
                emp_color='red',
                elev_offset=-27,
                azim_offset=225,
                roll_offset=0)

### F55 – Gallagher 101 peaks / Gallagher 101 peaks
bbob f21 / f21. PF empírica (NSGA-II).

In [ ]:
analyze_problem(BBOB_F55_Gallagher101_Gallagher101(), 'BBOB55',
                pair_vars=[1, 3, 5, 6],
                true_color='white',
                emp_color='red',
                elev_offset=-27,
                azim_offset=225,
                roll_offset=0)